In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0
from torchvision.models.efficientnet import SqueezeExcitation
import matplotlib.pyplot as plt
import multiprocessing
import numpy as np
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,  roc_auc_score,
    cohen_kappa_score, matthews_corrcoef, confusion_matrix, average_precision_score, fbeta_score,
    precision_recall_curve, balanced_accuracy_score)
import pandas as pd
from sklearn.utils import shuffle
import random
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
import torch_optimizer
import cv2
from skimage import measure
from torchvision.utils import make_grid
%matplotlib inline

In [ ]:
# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
!nvidia-smi

In [ ]:
file_name = "B0_HCAM_AdvAug(0.5,0.5)LAB_NoSDO_CV_Small_F7"
K = 2
Ft = 7
augmentation_options = ["lesionmix_color", "intensitymix", "none"]
augmentation_probs = [0.5, 0.5, 0]
dropout_rate = 0
color_space_options = ["Lab", "HSV"]
color_space_probs = [1, 0]

In [ ]:
import os

# Define directories of where the datasets reside
base_dir = '/home/pxk409/skin_lesions_13k'

# Define directories of where the training, validation and test sets reside
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

# Define directories of where the benign and malignant images reside
train_benign_dir = os.path.join(base_dir, 'train/benign')
train_malignant_dir = os.path.join(base_dir, 'train/malignant')
test_benign_dir = os.path.join(base_dir, 'test/benign')
test_malignant_dir = os.path.join(base_dir, 'test/malignant')

In [ ]:
# Let's check the number of images in each set
print('Total training benign images:', len(os.listdir(train_benign_dir)))
print('Total training malignant images:', len(os.listdir(train_malignant_dir)))
print('Total test benign images:', len(os.listdir(test_benign_dir)))
print('Total test malignant images:', len(os.listdir(test_malignant_dir)))

In [ ]:
# Load full dataset
full_dataset = datasets.ImageFolder(root=train_dir)

# Extract labels from the dataset
#labels = [label for _, label in full_dataset.samples]

# Stratified split (80% train, 20% validation)
#split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=123456)
#train_idx, val_idx = next(split.split(np.zeros(len(labels)), labels))

# Create training and validation subsets
#train_dataset = Subset(full_dataset, train_idx)
#val_dataset = Subset(full_dataset, val_idx)

#print(f"Training set: {len(train_idx)} samples")
#print(f"Validation set: {len(val_idx)} samples")

# Count the number of samples per class in each split
#train_classes = [full_dataset.targets[idx] for idx in train_dataset.indices]
#val_classes = [full_dataset.targets[idx] for idx in val_dataset.indices]

#print(f"Training set class counts: {np.bincount(train_classes)}")
#print(f"Validation set class counts: {np.bincount(val_classes)}")

In [ ]:
# Custom transform to integrate Albumentations with PyTorch
class AlbumentationsTransform:
    def __init__(self, augmentations):
        self.augmentations = A.Compose(augmentations)

    def __call__(self, image):
        image = np.array(image)  # Convert PIL Image to NumPy array
        augmented = self.augmentations(image=image)
        return augmented["image"]

# Simple augmentations for training
simple_transform = AlbumentationsTransform([
    A.RandomResizedCrop(height=224, width=224, scale=(0.8, 1.0), p=1.0),  # Random crop and resize
    A.HorizontalFlip(p=0.5),  # Horizontal flip
    A.VerticalFlip(p=0.5),  # Vertical flip
    #A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),  # Color jitter
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize to match pre-trained model
    ToTensorV2(),  # Convert to PyTorch tensor and ensure dtype=torch.float32
])

# Advanced augmentations for training
advanced_transform = AlbumentationsTransform([
    A.ElasticTransform(alpha=1.0, p=0.5),  # Elastic deformation
    A.CLAHE(clip_limit=2.0, p=0.5),  # Contrast Limited Adaptive Histogram Equalization
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, min_holes=1, min_height=8, min_width=8, p=0.5),  # Coarse dropout
    A.GaussianBlur(blur_limit=(3, 7), p=0.5),  # Gaussian blur
    A.RandomResizedCrop(height=224, width=224, scale=(0.8, 1.0), p=1.0),  # Random crop and resize
    A.HorizontalFlip(p=0.5),  # Horizontal flip
    A.VerticalFlip(p=0.5),  # Vertical flip
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),  # Color jitter
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize to match pre-trained model
    ToTensorV2(),  # Convert to PyTorch tensor and ensure dtype=torch.float32
])

# Use AlbumentationsTransform directly in the loaders for consistency
train_transform = simple_transform
#train_transform = advanced_transform

# Albumentations augmentations for validation and test
albumentations_val_test_transform = AlbumentationsTransform([
    A.Resize(height=224, width=224),  # Resize to EfficientNet input size
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize to match pre-trained model
    ToTensorV2()  # Convert to PyTorch tensor
])

test_val_transform = albumentations_val_test_transform


In [ ]:
class TransformDataset(Dataset):
    def __init__(self, dataset, transform):
        """
        Args:
            dataset: Original dataset (e.g., Subset of ImageFolder).
            transform: Transform to be applied to the images in the dataset.
        """
        self.dataset = dataset
        self.transform = transform

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.dataset)

In [ ]:
# Wrap subsets with the respective transforms
#train_dataset = TransformDataset(train_dataset, train_transform)
#val_dataset = TransformDataset(val_dataset, test_val_transform)

In [ ]:
# Load and prepare the test dataset
#test_dataset = datasets.ImageFolder(root=test_dir, transform=test_val_transform)
test_dataset = datasets.ImageFolder(root=test_dir)
test_dataset = TransformDataset(test_dataset, test_val_transform)

In [ ]:
# Create data loaders
batch_size = 64
#num_cpus = multiprocessing.cpu_count()  # Use all CPU cores
#print(f"Number of CPU cores: {num_cpus}")
#num_workers = 16


#train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
#test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
#train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
#val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
#test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

In [ ]:
# LesionMix Implementation with Color-Space Transformation

def denormalize(x, mean, std):
    """Denormalize normalized images."""
    mean = torch.tensor(mean).view(1, 3, 1, 1).to(x.device)
    std = torch.tensor(std).view(1, 3, 1, 1).to(x.device)
    return x * std + mean

def color_space_thresholding(x, color_space='Lab', threshold_factor=0.5):
    """
    Apply color-space transformation and thresholding to generate lesion masks.
    Handles normalized images by denormalizing them first.

    Args:
        x (torch.Tensor): Batch of images (N, C, H, W) in normalized RGB format.
        color_space (str): 'Lab' or 'HSV' for color-space transformation.
        threshold_factor (float): Multiplier for the threshold.

    Returns:
        torch.Tensor: Binary masks for lesion regions (N, 1, H, W).
    """
    # Denormalize the image
    x_denorm = denormalize(x, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    x_denorm_np = (x_denorm.permute(0, 2, 3, 1).cpu().numpy() * 255).astype(np.uint8)  # Convert to NumPy and scale to [0, 255]

    masks = []
    for img in x_denorm_np:
        if color_space == 'Lab':
            # Convert to Lab color space
            lab_img = cv2.cvtColor(img, cv2.COLOR_RGB2Lab)
            l_channel = lab_img[:, :, 0]  # Extract luminance (L) channel
            threshold = l_channel.mean() + l_channel.std() * threshold_factor
            mask = (l_channel > threshold).astype(np.float32)  # Apply threshold

        elif color_space == 'HSV':
            # Convert to HSV color space
            hsv_img = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
            s_channel = hsv_img[:, :, 1]  # Extract saturation (S) channel
            threshold = s_channel.mean() + s_channel.std() * threshold_factor
            mask = (s_channel > threshold).astype(np.float32)  # Apply threshold

        else:
            raise ValueError("Invalid color_space. Choose 'Lab' or 'HSV'.")

        masks.append(mask)

    # Convert masks back to torch.Tensor
    masks = torch.tensor(np.stack(masks), device=x.device).unsqueeze(1)
    return masks

def lesionmix_data_color_space(x, y, alpha=1.0, color_space='Lab'):
    """
    LesionMix data augmentation with color-space thresholding.

    Args:
        x (torch.Tensor): Batch of images (N, C, H, W) in normalized RGB format.
        y (torch.Tensor): Batch of labels.
        alpha (float): MixUp interpolation parameter.
        color_space (str): 'Lab' or 'HSV' for color-space transformation.

    Returns:
        mixed_x: Mixed input images.
        y_a: Original labels.
        y_b: Labels from mixed images.
        lam: Interpolation factor.
    """
    if alpha > 0.0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    # Extract lesion region using color-space thresholding
    masks = color_space_thresholding(x, color_space=color_space)
    lesion_x = masks * x
    lesion_x_mixed = lam * lesion_x + (1 - lam) * lesion_x[index, :]

    mixed_x = lam * x + (1 - lam) * x[index, :]
    mixed_x = torch.where(masks > 0, lesion_x_mixed, mixed_x)

    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def lesionmix_criterion(criterion, pred, y_a, y_b, lam):
    """loss function."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
# IntensityMix Implementation

def gaussian_weighting(mask, sigma=5):
    """
    Generate a Gaussian weight map centered on the lesion area.
    Args:
        mask: Binary lesion mask (0 or 1).
        sigma: Standard deviation for Gaussian blur.
    Returns:
        Gaussian-weighted mask.
    """
    device = mask.device
    mask_np = mask.cpu().numpy().astype('float32')
    weighted_masks = []
    for m in mask_np:
        blurred = cv2.GaussianBlur(m, (sigma * 2 + 1, sigma * 2 + 1), sigma)
        weighted_masks.append(blurred / blurred.max())  # Normalize to [0, 1]
    return torch.tensor(np.stack(weighted_masks), device=device)

def intensitymix_data(x, y, alpha=1.0, color_space='Lab'):
    """
    IntensityMix data augmentation with color-space-based thresholding for lesion regions.
    Args:
        x: Input batch of images (normalized).
        y: Corresponding labels.
        alpha: Mix factor for intensity blending.
        color_space: Color space to use ('Lab' or 'HSV').
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    # Convert to the specified color space
    x_denorm = denormalize(x, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    x_np = x_denorm.permute(0, 2, 3, 1).cpu().numpy()  # Convert to [B, H, W, C]

    masks = []
    for img in x_np:
        if color_space == 'Lab':
            img_lab = cv2.cvtColor((img * 255).astype('uint8'), cv2.COLOR_RGB2Lab)
            l_channel = img_lab[:, :, 0]  # Extract lightness channel
            _, mask = cv2.threshold(l_channel, 128, 255, cv2.THRESH_BINARY)
        elif color_space == 'HSV':
            img_hsv = cv2.cvtColor((img * 255).astype('uint8'), cv2.COLOR_RGB2HSV)
            v_channel = img_hsv[:, :, 2]  # Extract value channel
            _, mask = cv2.threshold(v_channel, 128, 255, cv2.THRESH_BINARY)
        else:
            raise ValueError("Invalid color_space. Choose 'Lab' or 'HSV'.")
        masks.append(torch.tensor(mask / 255.0, device=x.device))

    # Stack masks and reshape
    masks = torch.stack(masks).unsqueeze(1).float()  # Shape: [B, 1, H, W]

    # Generate mixed regions based on intensity blending
    lesion_x = masks * x
    lesion_x_mixed = lam * lesion_x + (1 - lam) * lesion_x[index, :]
    x_mixed = lam * x + (1 - lam) * x[index, :]
    x_mixed = torch.where(masks > 0, lesion_x_mixed, x_mixed)

    y_a, y_b = y, y[index]
    return x_mixed, y_a, y_b, lam


def intensitymix_criterion(criterion, pred, y_a, y_b, lam):
    """
    IntensityMix loss function.
    Args:
        criterion: Loss function.
        pred: Model predictions.
        y_a: First set of labels.
        y_b: Second set of labels.
        lam: Blending factor.
    Returns:
        Loss value.
    """
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


In [ ]:
# Channel Attention
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio):
        super(ChannelAttention, self).__init__()
        # Global descriptors from avg and max pooling
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        # Shared MLP for attention generation
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction_ratio, bias=False),
            nn.ReLU(),
            nn.Linear(in_channels // reduction_ratio, in_channels, bias=False)
        )

        # Activation to produce the attention map
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()

        # Generate global descriptors
        avg_out = self.fc(self.avg_pool(x).view(b, c))  # Avg pooling path
        max_out = self.fc(self.max_pool(x).view(b, c))  # Max pooling path

        # Combine and produce channel attention
        out = avg_out + max_out
        return self.sigmoid(out).view(b, c, 1, 1)  # Reshape for channel-wise multiplication


# Global Context Block
class GlobalContextBlock(nn.Module):
    def __init__(self, in_channels, reduction_ratio):
        super(GlobalContextBlock, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction_ratio, bias=False),
            nn.ReLU(),
            nn.Linear(in_channels // reduction_ratio, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, h, w = x.size()
        # Compute global context vector
        context = x.mean(dim=(2, 3), keepdim=False)  # Global average pooling
        context = self.fc(context)  # Apply FC layers
        return x * context.view(b, c, 1, 1)  # Broadcast context to match spatial dimensions


# Multi-Scale Spatial Attention
class MultiScaleSpatialAttention(nn.Module):
    def __init__(self, kernel_sizes=[3, 5, 7, 9]):
        super(MultiScaleSpatialAttention, self).__init__()
        self.spatial_convs = nn.ModuleList([
            nn.Conv2d(2, 1, kernel_size=ks, padding=(ks - 1) // 2, bias=False) # Same padding
            for ks in kernel_sizes
        ])
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)  # Avg pooling path
        max_out, _ = torch.max(x, dim=1, keepdim=True)  # Max pooling path
        combined = torch.cat([avg_out, max_out], dim=1)  # Combine features

        # Apply each spatial attention head
        multi_scale_outputs = [self.sigmoid(conv(combined)) for conv in self.spatial_convs]
        # Combine multi-scale outputs
        out = torch.sum(torch.stack(multi_scale_outputs, dim=0), dim=0)
        return out * x  # Apply spatial attention


# Spatial Dropout Class
class SpatialDropout(nn.Module):
    def __init__(self, p=dropout_rate):
        super(SpatialDropout, self).__init__()
        self.p = p

    def forward(self, x):
        # Apply channel-wise dropout
        return nn.functional.dropout2d(x, self.p, training=self.training)



# HCAM Block
class HCAM(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16, kernel_sizes=[3, 5, 7, 9]):
        super(HCAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction_ratio)
        self.global_context_block = GlobalContextBlock(in_channels, reduction_ratio)
        self.spatial_attention = MultiScaleSpatialAttention(kernel_sizes)

    def forward(self, x):
        # Apply channel attention
        out = self.channel_attention(x) * x
        # Apply global context block
        out = self.global_context_block(out)
        # Apply multi-scale spatial attention
        out = self.spatial_attention(out)
        return out


# Replace SE blocks with HCAM and add spatial dropout
def replace_se_with_hcam(module, dropout_rate=dropout_rate):
    for name, layer in module._modules.items():
        if isinstance(layer, SqueezeExcitation):
            in_channels = layer.fc1.in_channels
            # Replace SE block with HCAM and add Spatial Dropout
            module._modules[name] = nn.Sequential(
                HCAM(in_channels=in_channels)
            )
            #print(f"Replaced SE block with Enhanced CBAM and added Spatial Dropout in: {name}")
        elif isinstance(layer, nn.Sequential) or isinstance(layer, nn.Module):
            replace_se_with_hcam(layer, dropout_rate)


In [ ]:
def initialize_model(K=2):
    # Load pretrained EfficientNet-B0
    model = efficientnet_b0(weights="DEFAULT")

    # Select only the last K layers of model.features
    last_k_layers = list(model.features.children())[-K:]

    # Replace SE blocks with Enhanced CBAM in the last K layers
    replace_se_with_hcam(nn.Sequential(*last_k_layers))

    # Add spatial dropout
    #model.features[7].add_module("spatial_dropout", SpatialDropout(p=dropout_rate))
    #model.features[8].add_module("spatial_dropout", SpatialDropout(p=dropout_rate))

    # Freeze weights for the remaining layers (first len(features) - K layers)
    for layer in list(model.features.children())[:-K]:
        for param in layer.parameters():
            param.requires_grad = False

    # Modify the classifier for binary classification
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.classifier[1].in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 2)
    )

    # Move model to device
    model = model.to(device)
    return model

model = initialize_model()
print(model)

In [ ]:
# Define loss function and optimizer
#criterion = nn.CrossEntropyLoss()
#optimizer = torch_optimizer.AdamP(model.parameters(), lr=0.001)
#optimizer = torch_optimizer.Ranger(model.parameters(), lr=0.001)
#optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Function to count trainable and non-trainable parameters
def count_trainable_non_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return trainable, non_trainable

# Count parameters in the convolutional part
conv_trainable, conv_non_trainable = count_trainable_non_trainable_parameters(model.features)
print(f"Convolutional part (features):\n"
      f"  Trainable parameters: {conv_trainable:,}\n"
      f"  Non-trainable parameters: {conv_non_trainable:,}")

# Count parameters in the classifier part
classifier_trainable, classifier_non_trainable = count_trainable_non_trainable_parameters(model.classifier)
print(f"Classifier part:\n"
      f"  Trainable parameters: {classifier_trainable:,}\n"
      f"  Non-trainable parameters: {classifier_non_trainable:,}")

# Count parameters in the entire model
total_trainable, total_non_trainable = count_trainable_non_trainable_parameters(model)
print(f"Whole model:\n"
      f"  Trainable parameters: {total_trainable:,}\n"
      f"  Non-trainable parameters: {total_non_trainable:,}")

In [ ]:
# Early Stopping
class EarlyStopping:
    def __init__(self, patience, delta, path):
        """
        Args:
            patience (int): Number of epochs to wait after last improvement.
            delta (float): Minimum change in monitored metric to qualify as an improvement.
            path (str): Path to save the best model.
        """
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        score = -val_loss  # Early stopping is based on minimizing validation loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        """Saves model when validation loss decreases."""
        print(f"Validation loss decreased. Saving model to {self.path}")
        torch.save(model.state_dict(), self.path)


# Set the path for saving best_model.pth
#current_dir = os.getcwd()
#model_save_path = os.path.join(current_dir, 'best_model2.pth')

# Initialize EarlyStopping
#early_stopping = EarlyStopping(patience=5, delta=0.001, path=model_save_path)

In [ ]:
# Helper function to evaluate a dataset
def evaluate_model(model, data_loader, device):
    model.eval()  # Set model to evaluation mode
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]  # Probabilities for the positive class
            _, preds = torch.max(outputs, 1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

# Helper function to calculate selected metrics
def calculate_metrics(labels, preds, probs):
    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds)
    recall = recall_score(labels, preds)
    f1 = f1_score(labels, preds)
    f2 = fbeta_score(labels, preds, beta=2)
    auc = roc_auc_score(labels, probs)
    specificity = recall_score(labels, preds, pos_label=0)
    npv = precision_score(labels, preds, pos_label=0)

    return recall, precision, accuracy, f1, f2, auc, specificity, npv

In [ ]:
# Training loop with 5-fold cross-validation
cv_results = []
cv_metrics = []
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=123456)

# Extract labels from the dataset
labels = [label for _, label in full_dataset.samples]

# Cross-validation loop
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), 1):
    print(f"\n========== Fold {fold} ==========\n")

    # Create training and validation subsets for this fold
    train_dataset = Subset(full_dataset, train_idx)
    val_dataset = Subset(full_dataset, val_idx)

    # Apply transformations
    train_dataset = TransformDataset(train_dataset, train_transform)
    val_dataset = TransformDataset(val_dataset, test_val_transform)

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model and optimizer for each fold
    model = initialize_model(K=K)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # EarlyStopping initialization
    early_stopping = EarlyStopping(patience=10, delta=0.0001, path=os.path.join(os.getcwd(), f"{file_name}_fold_{fold}.pth"))

    # Training loop
    num_epochs = 100
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Select augmentation type
            augmentation_type = np.random.choice(augmentation_options, p=augmentation_probs)
            color_space = np.random.choice(color_space_options, p=color_space_probs)

            if augmentation_type == "lesionmix_color":
                images, labels_a, labels_b, lam = lesionmix_data_color_space(images, labels, alpha=1.0, color_space=color_space)
                outputs = model(images)
                loss = lesionmix_criterion(criterion, outputs, labels_a, labels_b, lam)

            elif augmentation_type == "intensitymix":
                images, labels_a, labels_b, lam = intensitymix_data(images, labels, alpha=1.0, color_space=color_space)
                outputs = model(images)
                loss = intensitymix_criterion(criterion, outputs, labels_a, labels_b, lam)

            else:
                # No augmentation
                outputs = model(images)
                loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            # Calculate weighted accuracy for augmented data
            if augmentation_type in ["lesionmix_color", "intensitymix"]:
                train_correct += (lam * (preds == labels_a).sum().float() +
                                  (1 - lam) * (preds == labels_b).sum().float()).long()
            else:
                train_correct += torch.sum(preds == labels)

        train_loss /= len(train_loader)
        train_accuracy = train_correct.double() / len(train_dataset)
        train_losses.append(train_loss)

        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels)

        val_loss /= len(val_loader)
        val_accuracy = val_correct.double() / len(val_dataset)
        val_losses.append(val_loss)

        print(f"Epoch {epoch + 1}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

        # Call EarlyStopping
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping triggered. Ending training.")
            break

    # Store results for this fold
    cv_results.append({
        "fold": fold,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy.item(),
        "val_loss": val_loss,
        "val_accuracy": val_accuracy.item()
        })

    # Load the best model for evaluation
    model.load_state_dict(torch.load(f"{file_name}_fold_{fold}.pth", weights_only=True))

    # Evaluate on validation set
    labels, preds, probs = evaluate_model(model, val_loader, device)
    metrics = calculate_metrics(labels, preds, probs)
    recall, precision, accuracy, f1, f2, auc, specificity, npv = metrics

    # Record metrics for the fold
    cv_metrics.append({
        "Fold": fold,
        "Recall": recall,
        "Precision": precision,
        "Accuracy": accuracy,
        "F1 Score": f1,
        "F2 Score": f2,
        "AUC": auc,
        "Specificity": specificity,
        "NPV": npv,
    })

# Summarize cross-validation results
df_cv_results = pd.DataFrame(cv_results)
print("\n========== Cross-Validation Results ==========\n")
print(df_cv_results)

print("\nAverage Validation Accuracy: {:.4f}".format(df_cv_results["val_accuracy"].mean()))

In [ ]:
# Convert results to DataFrame
cv_metrics_df = pd.DataFrame(cv_metrics)

# Calculate average metrics
avg_metrics = cv_metrics_df.mean().to_dict()
avg_metrics["Fold"] = "Average"

# Add the average metrics to the DataFrame
cv_metrics_df = cv_metrics_df.append(avg_metrics, ignore_index=True)

# Save results to CSV file
csv_file_name = f"HCAM_CV_Results/{file_name}.csv"
cv_metrics_df.to_csv(csv_file_name, index=False)

print(f"Cross-validation metrics saved to {csv_file_name}")

In [ ]:
#for i, layer in enumerate(model.features):
#    print(f"Layer {i}: {layer}")


In [ ]:
#from torchsummary import summary
#summary(model, input_size=(3, 224, 224))

In [ ]:
def fine_tune_model(Ft, file_name, fold):
    """
    Fine-tunes the model by unfreezing the last Ft layers of the convolutional part.

    Args:
        Ft (int): Number of convolutional layers to unfreeze.
        file_name (str): File name of the saved model.
        fold (int): Current fold index.

    Returns:
        model: Fine-tuned model.
    """
    # Load the best model from the current fold
    model = initialize_model(K=K)
    model.load_state_dict(torch.load(f"{file_name}_fold_{fold}.pth", weights_only=True))

    # Freeze all convolutional layers
    for param in model.features.parameters():
        param.requires_grad = False

    # Unfreeze the last Ft layers of the convolutional part
    layers_to_unfreeze = list(model.features.children())[-Ft:]
    for layer in layers_to_unfreeze:
        for param in layer.parameters():
            param.requires_grad = True

    # Move model to device
    model = model.to(device)
    return model


In [ ]:
# Function to print trainability status of layers
model = fine_tune_model(Ft=Ft, file_name=file_name, fold=1)

def list_layer_status(model):
    print(f"{'Layer':<60} {'Trainable':<10}")
    print("=" * 70)
    for name, param in model.named_parameters():
        trainable_status = "Unfrozen" if param.requires_grad else "Frozen"
        print(f"{name:<60} {trainable_status}")

# Call the function for the entire model
list_layer_status(model)

In [ ]:
# Early Stopping
class EarlyStopping:
    def __init__(self, patience, delta, path):
        """
        Args:
            patience (int): Number of epochs to wait after last improvement.
            delta (float): Minimum change in monitored metric to qualify as an improvement.
            path (str): Path to save the best model.
        """
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_accuracy, model):
        """
        Args:
            val_accuracy (float): Validation accuracy for the current epoch.
            model: The model to save if validation accuracy improves.
        """
        score = val_accuracy

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_accuracy, model)
        elif score < self.best_score - self.delta:  # Check if accuracy decreased more than the delta
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_accuracy, model)
            self.counter = 0

    def save_checkpoint(self, val_accuracy, model):
        """Saves model when validation accuracy increases."""
        print(f"Validation accuracy improved. Saving model to {self.path}")
        torch.save(model.state_dict(), self.path)


In [ ]:
# Fine-Tuning with Cross-Validation
cv_results = []
cv_metrics = []
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=123456)

# Extract labels from the dataset
labels = [label for _, label in full_dataset.samples]

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), 1):
    print(f"\n========== Fine-Tuning: Fold {fold} ==========\n")

    # Create training and validation subsets for this fold
    train_dataset = Subset(full_dataset, train_idx)
    val_dataset = Subset(full_dataset, val_idx)

    # Apply transformations
    train_dataset = TransformDataset(train_dataset, train_transform)
    val_dataset = TransformDataset(val_dataset, test_val_transform)

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model and optimizer
    model = fine_tune_model(Ft=6, file_name=file_name, fold=fold)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)  # Smaller LR for fine-tuning

    # EarlyStopping initialization
    early_stopping = EarlyStopping(patience=50, delta=0.00001, path=os.path.join(os.getcwd(), f"{file_name}_fold_{fold}_Ft.pth"))

    if fold == 5:
        # Fine-Tuning Loop
        num_epochs = 1000
        for epoch in range(num_epochs):
            # Training phase
            model.train()
            train_loss = 0
            train_correct = 0

            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)

                # Select augmentation type
                augmentation_type = np.random.choice(augmentation_options, p=augmentation_probs)
                color_space = np.random.choice(color_space_options, p=color_space_probs)

                if augmentation_type == "lesionmix_color":
                    images, labels_a, labels_b, lam = lesionmix_data_color_space(images, labels, alpha=1.0, color_space=color_space)
                    outputs = model(images)
                    loss = lesionmix_criterion(criterion, outputs, labels_a, labels_b, lam)

                elif augmentation_type == "intensitymix":
                    images, labels_a, labels_b, lam = intensitymix_data(images, labels, alpha=1.0, color_space=color_space)
                    outputs = model(images)
                    loss = intensitymix_criterion(criterion, outputs, labels_a, labels_b, lam)

                else:
                    # No augmentation
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                # Backward pass and optimization
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                _, preds = torch.max(outputs, 1)

                # Calculate weighted accuracy for augmented data
                if augmentation_type in ["lesionmix_color", "intensitymix"]:
                    train_correct += (lam * (preds == labels_a).sum().float() +
                                      (1 - lam) * (preds == labels_b).sum().float()).long()
                else:
                    train_correct += torch.sum(preds == labels)

            train_loss /= len(train_loader)
            train_accuracy = train_correct.double() / len(train_dataset)

            # Validation phase
            model.eval()
            val_loss = 0
            val_correct = 0

            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)

                    outputs = model(images)
                    loss = criterion(outputs, labels)

                    val_loss += loss.item()
                    _, preds = torch.max(outputs, 1)
                    val_correct += torch.sum(preds == labels)

            val_loss /= len(val_loader)
            val_accuracy = val_correct.double() / len(val_dataset)

            print(f"Epoch {epoch + 1}/{num_epochs} | "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} | "
                  f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

            # Early Stopping
            early_stopping(val_accuracy.item(), model)
            if early_stopping.early_stop:
                print("Early stopping triggered. Ending training.")
                break

    # Load the best model for evaluation
    model.load_state_dict(torch.load(f"{file_name}_fold_{fold}_Ft.pth", weights_only=True))

    # Evaluate on validation set
    labels, preds, probs = evaluate_model(model, val_loader, device)
    metrics = calculate_metrics(labels, preds, probs)
    recall, precision, accuracy, f1, f2, auc, specificity, npv = metrics

    # Record metrics for the fold
    cv_metrics.append({
        "Fold": fold,
        "Recall": recall,
        "Precision": precision,
        "Accuracy": accuracy,
        "F1 Score": f1,
        "F2 Score": f2,
        "AUC": auc,
        "Specificity": specificity,
        "NPV": npv,
    })

    if fold == 5:
        # Store results for this fold
        cv_results.append({
            "fold": fold,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy.item(),
            "val_loss": val_loss,
            "val_accuracy": val_accuracy.item()
        })


In [ ]:
# Convert results to DataFrame
cv_metrics_df = pd.DataFrame(cv_metrics)

# Calculate average metrics
avg_metrics = cv_metrics_df.mean().to_dict()
avg_metrics["Fold"] = "Average"

# Add the average metrics to the DataFrame
cv_metrics_df = cv_metrics_df.append(avg_metrics, ignore_index=True)

# Save results to CSV file
csv_file_name = f"HCAM_CV_Results/{file_name}_Ft.csv"
cv_metrics_df.to_csv(csv_file_name, index=False)

print(f"Cross-validation metrics after fine-tuning saved to {csv_file_name}")